In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\arktr\Downloads\supply_cahin_data.xlsx")

In [5]:
df.head(1)

,Shipment_ID,Supplier_ID,Warehouse_ID,Product_ID,Product_Category,Supplier_Name,Origin_City,Destination_City,Shipment_Date,Expected_Delivery,...,Carrier,Delivery_Status,Delay_Days,Warehouse_Stock,Reorder_Level,Supplier_Rating,Defect_Rate,Inspection_Status,Payment_Status,Region
0,SHP100001,SUP030,WH018,PRD0124,Automotive,Prime Manufacturing 20,Hyderabad,Hyderabad,2024-08-22 00:00:00,2024-08-30 00:00:00,...,DELHIVERY,Delivered,0,2356,1113,4.9,5.04,Failed,Paid,South


In [6]:
## understandig data
df.describe()
df.size
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6150 entries, 0 to 6149
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Shipment_ID        6150 non-null   object 
 1   Supplier_ID        6150 non-null   object 
 2   Warehouse_ID       6150 non-null   object 
 3   Product_ID         6150 non-null   object 
 4   Product_Category   6150 non-null   object 
 5   Supplier_Name      5996 non-null   object 
 6   Origin_City        6150 non-null   object 
 7   Destination_City   6028 non-null   object 
 8   Shipment_Date      6150 non-null   object 
 9   Expected_Delivery  6150 non-null   object 
 10  Actual_Delivery    4371 non-null   object 
 11  Quantity           6052 non-null   object 
 12  Unit_Cost          6047 non-null   object 
 13  Total_Cost         6150 non-null   float64
 14  Shipping_Mode      6150 non-null   object 
 15  Carrier            6028 non-null   object 
 16  Delivery_Status    6150 

In [8]:
## to find the number of missing values
df_missing=df.isna().sum()
print(df_missing[df_missing>0])

Supplier_Name         154
Destination_City      122
Actual_Delivery      1779
Quantity               98
Unit_Cost             103
Carrier               122
Warehouse_Stock       129
Supplier_Rating       124
Defect_Rate           122
Inspection_Status     122
Payment_Status        157
dtype: int64


In [12]:
## checking  number of duplicates
print("\n Duplicated Shipment ID:",df['Shipment_ID'].duplicated().sum())


 Duplicated Shipment ID: 149


In [20]:
## checking unique values 
df_category=['Product_Category', 'Shipping_Mode', 'Carrier', 'Delivery_Status',
            'Inspection_Status', 'Payment_Status', 'Region']
for c in df_category:
    print(f"\nunique value in {c}({df[c].nunique()}):\n",df[c].unique())


unique value in Product_Category(30):
 ['Automotive' 'Pharmaceutical' 'Electronics' 'Textiles'
 'Industrial Equipment' 'Packaging' ' Electronics' ' Packaging '
 ' Automotive' 'Industrial Equipment ' 'Pharmaceutical ' 'Textiles '
 'PACKAGING' 'Electronics ' ' Pharmaceutical' 'Packaging ' 'ELECTRONICS'
 ' Textiles ' 'AUTOMOTIVE' ' Industrial Equipment' ' Textiles'
 'PHARMACEUTICAL' ' Packaging' ' Electronics ' 'INDUSTRIAL EQUIPMENT'
 ' Industrial Equipment ' 'TEXTILES' ' Automotive ' ' Pharmaceutical '
 'Automotive ']

unique value in Shipping_Mode(20):
 [' Sea' 'Sea' 'Road' 'Air' 'Rail' 'Air ' 'RAIL' ' Road ' ' Air' 'Sea '
 'SEA' 'Rail ' 'AIR' ' Rail' ' Air ' 'ROAD' 'Road ' ' Sea ' ' Road'
 ' Rail ']

unique value in Carrier(29):
 ['DELHIVERY' 'DHL' 'Safexpress' 'Delhivery' 'FedEx' 'BlueDart'
 'Local Carrier' ' FedEx' nan ' BlueDart' 'Delhivery ' ' Local Carrier '
 'LOCAL CARRIER' 'SAFEXPRESS' 'Local Carrier ' ' DHL ' 'FedEx ' ' DHL'
 'FEDEX' ' Delhivery ' 'BlueDart ' ' Local Carrier' 

In [23]:
## descriptive statistics
print("\n Numeric Summary:\n",df.describe(include=[np.number]))
print("\nCategorical summary:\n",df.describe(include=[object]))


 Numeric Summary:
          Total_Cost   Delay_Days  Reorder_Level  Supplier_Rating
count  6.150000e+03  6150.000000    6150.000000      6026.000000
mean   7.515639e+06     1.921626     793.853821         3.503419
std    6.575429e+06     2.327623     407.909508         0.882111
min    4.102370e+03     0.000000     100.000000        -1.000000
25%    2.014261e+06     0.000000     435.250000         2.800000
50%    5.661509e+06     1.000000     790.000000         3.500000
75%    1.159547e+07     3.000000    1151.000000         4.275000
max    2.923300e+07    13.000000    1500.000000         7.000000

Categorical summary:
        Shipment_ID Supplier_ID Warehouse_ID Product_ID Product_Category  \
count         6150        6150         6150       6150             6150   
unique        6001          80           20        250               30   
top      SHP105605      SUP057        WH012    PRD0121        Packaging   
freq             2          96          353         37             1029 

In [24]:
## creating duplicate data
df_clean=df.copy()

In [ ]:
## removing leading\trialling spaces in columns
df_clean.columns=df_clean.columns.str.strip()

In [25]:
## removing whitespaces in text/object type columns
text_col=df_clean.select_dtypes(include=['object']).columns.tolist()
for c in text_col:
    df_clean[c]=df_clean[c].astype(str).str.strip()
    df_clean.loc[df_clean[c].isin(['nan','none','NaT']),c]=np.nan

In [26]:
## statndardising text
std_col=['Product_Category', 'Supplier_Name', 'Origin_City',
                     'Destination_City', 'Shipping_Mode', 'Carrier',
                     'Delivery_Status', 'Inspection_Status', 'Payment_Status', 'Region']
for c in std_col:
    df_clean[c]=df_clean[c].str.title()

In [27]:
## converting  text_columns to numeric_columns
numeric_cols = ['Quantity', 'Unit_Cost', 'Warehouse_Stock', 'Defect_Rate']
for c in numeric_cols:
    df_clean[c]=pd.to_numeric(df_clean[c], errors='coerce')

In [28]:
## standardising date
df_clean['Shipment_Date']=pd.to_datetime(df_clean['Shipment_Date'], errors='coerce')
df_clean['Expected_Delivery']=pd.to_datetime(df_clean['Expected_Delivery'], errors='coerce')
df_clean['Actual_Delivery']=pd.to_datetime(df_clean['Actual_Delivery'], errors='coerce', format='mixed')

In [29]:
## handling invalid dates
bad_date_logic = df_clean['Actual_Delivery'] < df_clean['Shipment_Date']
df_clean.loc[bad_date_logic, 'Actual_Delivery'] = np.nan
bad_date2_logic=df_clean['Expected_Delivery']< df_clean['Shipment_Date']
df_clean.loc[bad_date2_logic,'Expected_Delivery']=np.nan

In [30]:
## Detect / remove invalid negative values
## Quantity, Unit_Cost, Warehouse_Stock, Total_Cost, Defect_Rate, Delay_Days can't be negative
non_negative_cols = ['Quantity', 'Unit_Cost', 'Warehouse_Stock', 'Total_Cost',
                      'Defect_Rate', 'Delay_Days']
for c in non_negative_cols:
    neg_cols=(df_clean[c]<0).sum()
    print(f"{c}:{neg_cols} negative values")
    df_clean.loc[df_clean[c]<0,c]=np.nan

Quantity:15 negative values
Unit_Cost:12 negative values
Warehouse_Stock:0 negative values
Total_Cost:0 negative values
Defect_Rate:0 negative values
Delay_Days:0 negative values


In [32]:
##handling missing values (numeric type data)
for c in ['Quantity', 'Unit_Cost', 'Warehouse_Stock', 'Defect_Rate', 'Supplier_Rating']:
    df_clean[c]=df_clean.groupby('Product_Category')[c].transform( lambda x :x.fillna(x.median()))
    df_clean[c]=df_clean[c].fillna(df_clean[c].median())


In [33]:
## fixing total cost
df_clean['Total_Cost']=df_clean['Quantity']*df_clean['Unit_Cost']

In [34]:
## fiiling categorical missing values
for c in ['Supplier_Name', 'Destination_City', 'Carrier', 'Inspection_Status', 'Payment_Status']:
    df_clean[c]=df_clean[c].fillna('Unknown')

In [35]:
## flagging actual delivery
df_clean['Is_Delivered']=df_clean['Actual_Delivery'].notna()

In [36]:
## removing duplictaes
before=len(df_clean)
print(before)

6150


In [37]:
df_clean=df_clean.drop_duplicates(subset='Shipment_ID', keep='first')
print(f"\n Removed {before-len(df_clean)} duplicate rows")


 Removed 150 duplicate rows


In [38]:
## cleaned data set validation
print("\n ---Validation---")
print("Shape after validation:",df_clean.shape)
print("Remaining nulls:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("Remaining negative value checks:",{c:(df_clean[c]<0).sum() for c in non_negative_cols})
print("Dtypes now:\n", df_clean.dtypes)
assert df_clean.duplicated().sum()==0, "Duplicates still present!"
print("Vaidation Passed.")


 ---Validation---
Shape after validation: (6000, 26)
Remaining nulls:
 Shipment_Date           5
Expected_Delivery     521
Actual_Delivery      2124
dtype: int64
Remaining negative value checks: {'Quantity': np.int64(0), 'Unit_Cost': np.int64(0), 'Warehouse_Stock': np.int64(0), 'Total_Cost': np.int64(0), 'Defect_Rate': np.int64(0), 'Delay_Days': np.int64(0)}
Dtypes now:
 Shipment_ID                  object
Supplier_ID                  object
Warehouse_ID                 object
Product_ID                   object
Product_Category             object
Supplier_Name                object
Origin_City                  object
Destination_City             object
Shipment_Date        datetime64[ns]
Expected_Delivery    datetime64[ns]
Actual_Delivery      datetime64[ns]
Quantity                    float64
Unit_Cost                   float64
Total_Cost                  float64
Shipping_Mode                object
Carrier                      object
Delivery_Status              object
Delay_Days   

In [41]:
##Product category analysis
print("\n Product Category Analysis:")
print(df_clean.groupby('Product_Category').agg(
    Total_Quantity=('Quantity','sum'),
    Total_Cost=('Total_Cost','sum'),
    Avg_Defect_Rate=('Defect_Rate','mean')
    ).sort_values('Total_Cost', ascending=False).round(2))


 Product Category Analysis:
                      Total_Quantity    Total_Cost  Avg_Defect_Rate
Product_Category                                                   
Packaging                  1047246.0  8.025017e+09             3.86
Industrial Equipment       1003692.0  7.848644e+09             4.11
Pharmaceutical              969054.5  7.389268e+09             4.25
Electronics                 997042.5  7.389072e+09             3.86
Textiles                    983950.0  7.294901e+09             4.06
Automotive                  979660.5  7.176950e+09             4.16


In [42]:
## regional analysis
print("\n Regional Analysis:")
print(df_clean.groupby('Region').agg(
    Total_Cost=('Total_Cost','sum'),
    Average_Delay=('Delay_Days','mean'),
    Shipment_Count=('Shipment_ID','count')).sort_values('Total_Cost',ascending=False).round(2))


 Regional Analysis:
          Total_Cost  Average_Delay  Shipment_Count
Region                                             
East    1.165609e+10           1.97            1540
West    1.142261e+10           1.81            1507
South   1.111650e+10           1.95            1497
North   1.092865e+10           1.95            1456


In [43]:
## cost analysis
print("Cost Analysis:")
print("Total Cost:\t",df_clean['Total_Cost'].sum())
print("Average unit cost by category:\n",df_clean.groupby('Product_Category')['Unit_Cost'].mean().round(2))

Cost Analysis:
Total Cost:	 45123852527.485
Average unit cost by category:
 Product_Category
Automotive              7415.75
Electronics             7364.77
Industrial Equipment    7887.65
Packaging               7559.06
Pharmaceutical          7574.53
Textiles                7474.06
Name: Unit_Cost, dtype: float64


In [44]:
## defect analysis
print("Defect Analysis:")
print(df_clean.groupby('Product_Category')['Defect_Rate'].mean().round(2).sort_values(ascending=False))

Defect Analysis:
Product_Category
Pharmaceutical          4.25
Automotive              4.16
Industrial Equipment    4.11
Textiles                4.06
Electronics             3.86
Packaging               3.86
Name: Defect_Rate, dtype: float64


In [45]:
## inspection analysis
print("Inspection anlysis:\n",df_clean['Inspection_Status'].value_counts())

Inspection anlysis:
 Inspection_Status
Failed     2010
Pending    1966
Passed     1904
Unknown     120
Name: count, dtype: int64


In [46]:
## monthly trend
df_clean['Shipment_Month'] = df_clean['Shipment_Date'].dt.to_period('M')
print("\nMonthly Trends")
monthly = df_clean.groupby('Shipment_Month').agg(
    shipments=('Shipment_ID', 'count'),
    total_cost=('Total_Cost', 'sum'),
    avg_delay=('Delay_Days', 'mean')
).round(2)
print(monthly)


Monthly Trends
                shipments    total_cost  avg_delay
Shipment_Month                                    
2024-01               193  1.460874e+09       2.12
2024-02               178  1.271956e+09       1.88
2024-03               212  1.440437e+09       1.79
2024-04               193  1.463661e+09       2.09
2024-05               213  1.562161e+09       2.03
2024-06               224  1.716563e+09       1.89
2024-07               222  1.859933e+09       1.83
2024-08               198  1.569348e+09       1.78
2024-09               200  1.617675e+09       2.06
2024-10               200  1.491118e+09       1.73
2024-11               183  1.253009e+09       2.19
2024-12               204  1.458581e+09       1.84
2025-01               218  1.689571e+09       1.83
2025-02               157  1.174016e+09       1.54
2025-03               198  1.507248e+09       2.01
2025-04               203  1.390711e+09       1.77
2025-05               240  1.641609e+09       2.01
2025-06        

In [47]:
# supplier analysis --- top 10 suplliers based on  avg supplier rating ---
print("\n Supplier Analysis:")
Supplier_Performance=df_clean.groupby('Supplier_Name').agg(
    Avg_Rating=('Supplier_Rating','mean'),
    Avg_Delays=('Delay_Days','mean'),
    Avg_Defect_Rate=('Defect_Rate','mean'),
    Total_Shipments=('Shipment_ID','count')).sort_values('Avg_Rating',ascending=False).round(2)
print(Supplier_Performance.head(10))


 Supplier Analysis:
                        Avg_Rating  Avg_Delays  Avg_Defect_Rate  \
Supplier_Name                                                     
Orion Industries 50           3.75        1.82             3.82   
Pacific Components 60         3.71        1.41             4.04   
Sigma Supplies 17             3.70        2.07             4.24   
Sigma Supplies 10             3.69        2.15             4.31   
Metro Manufacturing 39        3.69        1.88             3.86   
Pacific Components 22         3.67        2.00             3.46   
Eastern Traders 61            3.67        1.48             3.74   
Global Components 65          3.66        1.16             3.99   
Vertex Supplies 47            3.66        1.72             4.20   
Apex Industries 31            3.64        2.23             3.99   

                        Total_Shipments  
Supplier_Name                            
Orion Industries 50                  66  
Pacific Components 60                70  
Sigma 

In [48]:
## shipping mode analysis
print("\nShipping Mode Analysis")
print(df_clean.groupby('Shipping_Mode').agg(
    avg_cost=('Total_Cost', 'mean'),
    avg_delay=('Delay_Days', 'mean'),
    shipment_count=('Shipment_ID', 'count')
).round(2))


Shipping Mode Analysis
                 avg_cost  avg_delay  shipment_count
Shipping_Mode                                       
Air            7638254.52       1.90            1550
Rail           7310672.83       1.93            1454
Road           7047381.82       1.88            1508
Sea            8082922.00       1.97            1488


In [49]:
## carrier performance
print("\nCarrier Performance")
print(df_clean.groupby('Carrier').agg(
    avg_delay=('Delay_Days', 'mean'),
    on_time_rate=('Delay_Days', lambda s: (s <= 0).mean() * 100),
    shipment_count=('Shipment_ID', 'count')
).sort_values('avg_delay').round(2))


Carrier Performance
               avg_delay  on_time_rate  shipment_count
Carrier                                               
Dhl                 1.80         45.78             948
Delhivery           1.81         47.29            1032
Local Carrier       1.89         45.43             995
Unknown             1.93         43.33             120
Safexpress          1.94         46.65             939
Bluedart            2.02         44.88             976
Fedex               2.06         44.34             990


In [50]:
## using pivot table
print("\n [Pivot Table] avg dealy by region x shipping mode:")
pivot=pd.pivot_table(df_clean, index='Region', columns='Shipping_Mode', values='Delay_Days', aggfunc='mean').round(2)
print(pivot)


 [Pivot Table] avg dealy by region x shipping mode:
Shipping_Mode   Air  Rail  Road   Sea
Region                               
East           1.91  2.00  1.97  2.02
North          1.86  1.96  1.92  2.06
South          2.00  1.94  1.86  1.99
West           1.82  1.82  1.78  1.81


In [51]:
print("\n Quantity shipped in each product category:")
PC_Qty=pd.pivot_table(df_clean,index='Product_Category', values='Quantity', aggfunc='sum')
print(PC_Qty)


 Quantity shipped in each product category:
                       Quantity
Product_Category               
Automotive             979660.5
Electronics            997042.5
Industrial Equipment  1003692.0
Packaging             1047246.0
Pharmaceutical         969054.5
Textiles               983950.0


In [52]:
df_clean['Quantity']=df_clean['Quantity'].round().astype(int)

In [55]:
print("\n Quatity Shiped By Each Mode In Each Region:")
ccm=pd.pivot_table(df_clean, index='Shipping_Mode', columns='Region', values='Quantity', aggfunc='sum')
print(ccm)


 Quatity Shiped By Each Mode In Each Region:
Region           East   North   South    West
Shipping_Mode                                
Air            395360  368982  409066  397959
Rail           355728  367788  348951  361013
Road           365155  328129  368279  385462
Sea            434807  373589  353234  367157


In [56]:
## month over month change in shippment volume
monthly_sorted = monthly.sort_index()
monthly_sorted['Prev_Month_Shipments'] = monthly_sorted['shipments'].shift(1)
monthly_sorted['MoM_Change'] = monthly_sorted['shipments'] - monthly_sorted['Prev_Month_Shipments']
print("\n[shift] Month-over-month shipment volume change")
print(monthly_sorted[['shipments', 'Prev_Month_Shipments', 'MoM_Change']])


[shift] Month-over-month shipment volume change
                shipments  Prev_Month_Shipments  MoM_Change
Shipment_Month                                             
2024-01               193                   NaN         NaN
2024-02               178                 193.0       -15.0
2024-03               212                 178.0        34.0
2024-04               193                 212.0       -19.0
2024-05               213                 193.0        20.0
2024-06               224                 213.0        11.0
2024-07               222                 224.0        -2.0
2024-08               198                 222.0       -24.0
2024-09               200                 198.0         2.0
2024-10               200                 200.0         0.0
2024-11               183                 200.0       -17.0
2024-12               204                 183.0        21.0
2025-01               218                 204.0        14.0
2025-02               157                 218.0    